In [15]:
import polars as pl
import time

#1. Modo Lazy: Estamos apenas escaneando o arquivo, sem carregar para a RAM
#scan_parquet já inicia em modo lazy por padrão.
q = (
    pl.scan_parquet("../aula12/202502_NovoBolsaFamilia_POLARS.parquet")
    # .rename({
    #     'M�S COMPET�NCIA': 'MES_COMPETENCIA',
    #     'M�S REFER�NCIA': 'MES_REFERENCIA',
    #     'C�DIGO MUNIC�PIO SIAFI': 'CODIGO_MUNICIPIO_SIAFI',
    #     'NOME MUNIC�PIO': 'NOME_MUNICIPIO',
    # })
    # .filter(pl.col("UF") == "SP")
)

print(type(q)) # Note que é um LazyFrame, não um DataFrame
print("\n")

# 2. O Plano de Execução (Explain)
print("--- Plano de Execução (Explain) ---")
print("O Polars analisou sua query e otimizou o caminho antes de tocar nos dados:")
print(q.explain())
print("\n")

# 3. Execução Real (Collect)
print("--- Executando (Collect) ---")
start_time = time.time()
df_resultado = q.collect()
end_time = time.time()

display(df_resultado)

<class 'polars.lazyframe.frame.LazyFrame'>


--- Plano de Execução (Explain) ---
O Polars analisou sua query e otimizou o caminho antes de tocar nos dados:
Parquet SCAN [../aula12/202502_NovoBolsaFamilia_POLARS.parquet]
PROJECT */9 COLUMNS
ESTIMATED ROWS: 20500422


--- Executando (Collect) ---


M�S COMPET�NCIA,M�S REFER�NCIA,UF,C�DIGO MUNIC�PIO SIAFI,NOME MUNIC�PIO,CPF FAVORECIDO,NIS FAVORECIDO,NOME FAVORECIDO,VALOR PARCELA
i64,i64,str,i64,str,str,i64,str,str
202501,202308,"""SP""",7071,"""SANTOS""","""***.085.106-**""",20643890445,"""FERNANDA RAMOS TEIXEIRA""","""650,00"""
202501,202309,"""SP""",7071,"""SANTOS""","""***.085.106-**""",20643890445,"""FERNANDA RAMOS TEIXEIRA""","""650,00"""
202501,202310,"""SP""",7071,"""SANTOS""","""***.085.106-**""",20643890445,"""FERNANDA RAMOS TEIXEIRA""","""650,00"""
202501,202311,"""SP""",7071,"""SANTOS""","""***.085.106-**""",20643890445,"""FERNANDA RAMOS TEIXEIRA""","""650,00"""
202501,202312,"""SP""",7071,"""SANTOS""","""***.085.106-**""",20643890445,"""FERNANDA RAMOS TEIXEIRA""","""650,00"""
…,…,…,…,…,…,…,…,…
202501,202501,"""TO""",9643,"""XAMBIOA""","""""",16640890691,"""ZEIA DE SOUZA LUCIO""","""750,00"""
202501,202501,"""TO""",9643,"""XAMBIOA""","""***.273.191-**""",20644881997,"""ZENILDE ALVES DOS SANTOS""","""600,00"""
202501,202501,"""TO""",9643,"""XAMBIOA""","""""",19058661973,"""ZENOLIA RAMOS DA SILVA CARVALH…","""600,00"""


In [ ]:
# Definindo a consulta estatística
stats_query = (
    pl.scan_parquet("../Aula12/202501_NovoBolsaFamilia_POLARS.parquet")
    .with_columns(
        pl.col("VALOR PARCELA")
        .str.replace(",", ".")     # "650,00" vira "650.00"
        .cast(pl.Float64)          # vira o número 650.0
    )
    .select([
        pl.col("VALOR PARCELA").min().alias("minimo"),
        pl.col("VALOR PARCELA").mean().alias("media"),
        pl.col("VALOR PARCELA").std().alias("desvio_padrao"),
        pl.col("VALOR PARCELA").max().alias("maximo"),
        pl.col("VALOR PARCELA").quantile(0.5).alias("mediana")
    ])
)

# Executando
print("--- Estatísticas Descritivas ---")
stats_df = stats_query.collect()
display(stats_df)

: 